In [6]:
import pandas as pd

df = pd.read_csv("data/data.txt", sep="|", engine="python")
missing = df.isnull().sum().sort_values(ascending=False)
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    "MissingCount": missing,
    "MissingPercent": missing_percent
})

missing_df[missing_df["MissingCount"] > 0]


,MissingCount,MissingPercent
NumberOfVehiclesInFleet,1000098,100.000000
CrossBorder,999400,99.930207
CustomValueEstimate,779642,77.956560
Rebuilt,641901,64.183810
Converted,641901,64.183810
WrittenOff,641901,64.183810
NewVehicle,153295,15.327998
Bank,145961,14.594670
AccountType,40232,4.022806
Gender,9536,0.953507


In [7]:
df.drop(columns=[
    "NumberOfVehiclesInFleet",
    "CrossBorder",
    "Rebuilt",
    "Converted",
    "WrittenOff"
], inplace=True)
df["CustomValueEstimate"] = df["CustomValueEstimate"].fillna(
    df["CustomValueEstimate"].median()
)
df["NewVehicle"] = df["NewVehicle"].fillna("Unknown")
df["Bank"] = df["Bank"].fillna("Unknown")
df["AccountType"] = df["AccountType"].fillna("Unknown")
df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])
df["MaritalStatus"] = df["MaritalStatus"].fillna("Unknown")
num_cols = [
    "Cylinders",
    "kilowatts",
    "NumberOfDoors",
    "cubiccapacity"
]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
    cat_cols = ["VehicleType", "make", "Model", "bodytype"]




Missing values were handled using a structured approach based on the proportion of missingness and variable importance. Features with excessive missingness (>60%) were removed as they provided limited analytical value. Key predictive variables such as vehicle characteristics and insured value were retained and imputed using median values for numerical features and “Unknown” for categorical variables. This approach preserves data integrity while minimizing bias introduced by missing data.

In [12]:
df["VehicleAge"] = 2026 - df["RegistrationYear"]
df["TransactionMonth"] = pd.to_datetime(df["TransactionMonth"], errors="coerce")
df["PolicyYear"] = df["TransactionMonth"].dt.year
df["PolicyMonth"] = df["TransactionMonth"].dt.month
df["PowerToValue"] = df["kilowatts"] / df["CustomValueEstimate"]
df["EnginePowerRatio"] = df["kilowatts"] / df["cubiccapacity"]
df["PremiumToValueRatio"] = df["TotalPremium"] / df["SumInsured"]
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)
df["LossRatio"] = df["TotalClaims"] / df["TotalPremium"]
df.replace([float("inf"), -float("inf")], None, inplace=True)
num_cols = df.select_dtypes(include=["number"]).columns
cat_cols = df.select_dtypes(include=["object", "string"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna("Unknown")


Missing values were handled by applying type-specific imputation. Numerical features were filled using median values to reduce sensitivity to outliers, while categorical features were assigned an "Unknown" category to preserve information without introducing bias.

In [16]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns
cat_cols
df = pd.get_dummies(df, columns=[
    "Gender",
    "VehicleType",
    "MaritalStatus",
    "Bank",
    "AccountType"
], drop_first=True)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["Model"] = le.fit_transform(df["Model"].astype(str))
df["bodytype"] = le.fit_transform(df["bodytype"].astype(str))
df.select_dtypes(include=["object", "string"]).columns
df = pd.get_dummies(df, drop_first=True)

In [15]:
import sys
!{sys.executable} -m pip install  scikit-learn

  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-le